# IM19 — Intelligent Mind 19
Autonomous Knowledge Intelligence pilot. Colab runs only while its runtime is active. This notebook is unexecuted; it does not apply migrations or start a 24/7 worker. Use a reviewed staging database with the new migration and a restricted worker login. Never paste tokens into cells or outputs.

In [ ]:
!git clone https://github.com/StrategyG/RAG_Cerdas.git /content/RAG_Cerdas
%cd /content/RAG_Cerdas
!git checkout feat/im19-knowledge-intelligence-v2
!python -m pip install -q -r requirements-knowledge.txt
!apt-get -qq install tesseract-ocr tesseract-ocr-eng tesseract-ocr-ind

Add IM19_DATABASE_DSN, DROPBOX_APP_KEY, DROPBOX_APP_SECRET and DROPBOX_REFRESH_TOKEN in Colab Secrets. Use a staging DSN with sslmode=verify-full and a CA certificate. Download only reviewed local GGUF/E5 model files you intend to use; record hashes. No cloud LLM is used.

In [ ]:
from im19.secrets import ColabSecrets
from im19.storage.postgres import PostgresStore
from im19.knowledge_mining.dropbox_discovery import DropboxClient, DropboxDiscovery
secrets = ColabSecrets()
store = PostgresStore(secrets.get("IM19_DATABASE_DSN"))
store.check_role()
print("Staging database role checked; no production readiness claim.")

Set the UUID of a source connection created by the administrator for an approved app folder. This discovery writes metadata and a cursor into staging; it downloads no documents.

In [ ]:
CONNECTION_ID = "REPLACE_WITH_STAGING_SOURCE_CONNECTION_UUID"
if CONNECTION_ID.startswith("REPLACE"):
    raise ValueError("Set the approved staging source UUID first")
dropbox = DropboxClient(secrets)
report = DropboxDiscovery(dropbox, store).run(CONNECTION_ID, max_pages=1)
print(report)

Inspect staged metadata and authorize one source through the separate approver role. Set paths to already available local models. Human review decisions must reference the immutable review_run_id. The worker cannot approve itself.

In [ ]:
from im19.local_model import LocalQwen
from im19.retrieval import E5Embedder
from im19.orchestrator import Orchestrator
GGUF_PATH = "/content/models/qwen.gguf"
GGUF_SHA256 = "REPLACE_WITH_REVIEWED_GGUF_SHA256"
E5_PATH = "/content/models/multilingual-e5-small"
DOCUMENT_ID = "REPLACE_WITH_ONE_STAGING_DOCUMENT_UUID"
if DOCUMENT_ID.startswith("REPLACE") or GGUF_SHA256.startswith("REPLACE"):
    raise ValueError("Set one document UUID and a reviewed model hash")
model = LocalQwen(GGUF_PATH, model_sha256=GGUF_SHA256)
embedder = E5Embedder(E5_PATH)
runner = Orchestrator(store, dropbox, model, embedder, workdir="/content/im19-checkpoints", enable_ocr=True)
print({"job_claimed": runner.run_one(DOCUMENT_ID)})

After reviewing all five roles, provenance and source rights, the separate human approver may approve the review. Resume only that document. NOT_EXTERNALLY_VERIFIED remains unchanged. For recovery, persistent database checkpoints survive Colab disconnects; missing local downloads are fetched again by pinned revision.

In [ ]:
store.resume_approved()
print({"job_claimed": runner.run_one(DOCUMENT_ID)})
from im19.retrieval import HybridRetriever
answer = HybridRetriever(store, embedder).search("Apa prinsip manajemen risiko?", filters={"document_id": DOCUMENT_ID})
print(answer)